# Advanced object-oriented programming 🧠

## What you will learn in this course 🧐🧐

You understand classes, objects, attributes, and methods from the fundamentals lecture. These concepts work well for simple systems but operational complexity demands more sophisticated patterns. Real systems require hierarchies of related types, uniform interfaces for different implementations, and flexible composition of components.

By the end of this course, you will:

- Build inheritance hierarchies for specialized types
- Implement polymorphism for uniform interfaces
- Use composition to build complex systems from simple parts
- Apply abstraction to hide implementation complexity
- Integrate objects with Python's protocols through special methods

In [1]:
from datetime import date
from typing import List, Dict, Optional
from abc import ABC, abstractmethod

## Building ProPublica's content management system

<img src ="https://guides.propublica.org/design/assets/images/preview-main-wordmark.png" />

Your basic OOP skills created a working object management system. Now imagine you are the **lead developer at ProPublica** — the independent, non-profit investigative newsroom founded in 2008, funded entirely by foundations and reader donations with no advertising.

<Note title="Notable reporting and projects" type="info">

ProPublica has published many impactful investigations, including: "An Unbelievable Story of Rape". The piece won the Pulitzer Prize for Explanatory Reporting in 2016 and was adapted into the Netflix series "Unbelievable". ProPublica's "Dollars for Docs" database tracks payments from pharmaceutical companies to doctors, exposing conflicts of interest in healthcare. The "Electionland" project monitored voting issues during the 2016 US elections, providing real-time reporting on voter suppression and irregularities. ProPublica's "Documenting Hate" initiative collects data on hate crimes and bias incidents across the US, shedding light on underreported issues of discrimination. 

These projects demonstrate ProPublica's commitment to investigative journalism that holds power accountable and informs the public.

</Note>  

ProPublica publishes two very different types of content that must coexist in the same system:

- **Investigations**, long-form, source-verified reporting by staff journalists. These require FOIA requests, a minimum number of documented sources, a right-of-reply sent to all named subjects, and often a coordinated embargo with a partner outlet (NYT, Washington Post, etc.).
- **Data Reports**, data-driven pieces built on public datasets and scrapers. These go through a lighter editorial process but require a methodology statement and at least one peer review of the analysis before publication.

Both share common operations: `editorial review, status tracking, archiving` but follow completely different publication rules. The system must coordinate both through uniform interfaces. The procedural approach would duplicate review and access logic across every content type. Basic OOP would create independent classes, still duplicating shared behavior. 

In this lecture, you will learn how advanced OOP patterns solve these problems through inheritance, polymorphism, and composition.

## Abstraction and inheritance

<img src = "https://data-analytics-fullstack-assets.s3.eu-west-3.amazonaws.com/M05-Python_Programming/M2_D3_Abstraction.png"/>

### Abstraction

**Abstraction = focus on *what* an object can do, not *how* it does it.**

You define an interface like:

> Any `Content` at ProPublica can be `submitted_for_review()`, `archived()`, `published()`

Without caring (for now) whether it's a two-year investigation into tax avoidance by billionaires, or a data report built from public hospital records. From the outside, you just use these methods. The internal publication rules are hidden.

### Inheritance

**Inheritance = reuse + specialization.**

* You create a **base class** (e.g. `Content`) with common attributes and methods.
* Then you create **child classes** (e.g. `Investigation`, `DataReport`) that:

  * inherit everything from `Content`,
  * add or override behavior specific to their editorial rules.

So you avoid rewriting status tracking and review logic for every content type.

### Abstract base classes in Python

Python lets you formalize this idea with:

* `ABC` (Abstract Base Class)
* `@abstractmethod`

You define a class that says:

> Any subclass of me **must** implement these methods.

An abstract `Content` at ProPublica might require every child to implement `publish()` and `get_format()`.

It is literally an editorial contract:

> If you are a type of content on this platform, you promise to be publishable and to describe your format.

In [2]:
class Content(ABC):
    """Abstract base class: editorial contract for all ProPublica content

    ProPublica publishes everything under open access (free to read).
    The funding model is donations and foundations, not subscriptions or ads.
    """

    def __init__(self, title, author, word_count):
        """Initialize common content attributes"""
        self.title = title
        self.author = author
        self.word_count = word_count
        self.open_access = True    # all ProPublica content is free to read
        self.status = 'draft'      # draft → reviewed → published
        self.corrections = 0

    def submit_for_review(self):
        """Concrete method - shared across all content types"""
        if self.status == 'draft':
            self.status = 'reviewed'
        return f"'{self.title}' submitted for editorial review — status: {self.status}"

    def apply_correction(self, count=1):
        """Concrete method - track post-publication corrections"""
        if count < 0:
            raise ValueError("Correction count cannot be negative")
        self.corrections += count
        return self.corrections

    def is_ready(self):
        """Check if content has passed editorial review"""
        return self.status in ('reviewed', 'published')

    def is_published(self):
        """Check if content is already live"""
        return self.status == 'published'

    @abstractmethod
    def publish(self):
        """Abstract method — each content type has its own publication rules"""
        pass

    @abstractmethod
    def get_format(self):
        """Abstract method — type-specific editorial metadata"""
        pass

The `Investigation` class extends `Content` with the strict editorial requirements that define ProPublica's journalism: documented FOIA (Freedom of Information Act) requests, verified sources, right of reply sent to all named subjects, and embargo coordination with partner outlets.

<Note title="FOIA requests & embargoes" type="definition">

**FOIA requests** are formal applications to government agencies for access to public records. They are essential for investigative journalism, allowing reporters to obtain documents that reveal wrongdoing or inform the public. FOIA requests can take months to process and often require follow-up.

**Embargoes** are agreements between news organizations to hold back publication of a story until a specified date and time. This allows for coordinated releases, often with major outlets, maximizing impact and ensuring all parties have time to prepare. Embargoes are common for investigations that require careful timing, such as those involving legal proceedings or sensitive information.
</Note>

In [3]:
class Investigation(Content):
    """Long-form investigative report with strict pre-publication requirements"""

    def __init__(self, title, author, word_count, beat):
        # Call parent constructor — investigations are open access like all ProPublica content
        super().__init__(title, author, word_count)

        # Investigation-specific attributes
        self.beat = beat                        # e.g. 'health', 'criminal justice', 'finance'
        self.sources = []                       # list of verified source descriptions
        self.foia_requests = []                 # Freedom of Information Act requests filed
        self.right_of_reply_sent = False        # must be sent to all named subjects
        self.embargo_until = None               # hold publication until date (partner coordination)
        self.partner_outlet = None              # e.g. 'New York Times', 'Washington Post'
        self.page_views = 0

    def add_source(self, description):
        """Register a verified source for the investigation"""
        self.sources.append(description)
        return f"Source #{len(self.sources)} recorded: {description}"

    def file_foia(self, agency):
        """Log a FOIA request filed with a government agency"""
        self.foia_requests.append(agency)
        return f"📄 FOIA request filed with: {agency} ({len(self.foia_requests)} total)"

    def send_right_of_reply(self):
        """Mark that the right of reply has been sent to all named subjects"""
        self.right_of_reply_sent = True
        return f"✉️  Right of reply sent for: '{self.title}'"

    def set_embargo(self, until: date, partner: str):
        """Hold publication until a date, coordinated with a partner outlet"""
        self.embargo_until = until
        self.partner_outlet = partner
        return f"🔒 Embargo set until {until} — co-publishing with {partner}"

    def publish(self):
        """Implement abstract method — investigations have strict pre-publication rules"""
        if not self.is_ready():
            return f"❌ Cannot publish '{self.title}' — editorial review pending"
        if len(self.sources) < 2:
            return f"❌ Cannot publish '{self.title}' — at least 2 verified sources required"
        if not self.right_of_reply_sent:
            return f"❌ Cannot publish '{self.title}' — right of reply not yet sent"
        if self.embargo_until and self.embargo_until > date.today():
            return f"⏳ Embargo active until {self.embargo_until} — publication on hold"
        self.status = 'published'
        partner_tag = f" (co-published with {self.partner_outlet})" if self.partner_outlet else ""
        return f"📰 Investigation published [{self.beat}]: '{self.title}' by {self.author}{partner_tag}"

    def get_format(self):
        """Implement abstract method — investigation metadata"""
        return {
            'type': 'Investigation',
            'beat': self.beat,
            'sources': len(self.sources),
            'foia_requests': len(self.foia_requests),
            'right_of_reply': self.right_of_reply_sent,
            'embargo_until': str(self.embargo_until) if self.embargo_until else None,
            'partner': self.partner_outlet,
            'page_views': self.page_views
        }

    def __str__(self):
        embargo_tag = " 🔒" if self.embargo_until and not self.is_published() else ""
        return (f"Investigation{embargo_tag}: '{self.title}' [{self.beat}] "
                f"— {len(self.sources)} source(s), {len(self.foia_requests)} FOIA(s) "
                f"— {self.status}")

In [4]:
# Create an investigation inheriting from Content
story = Investigation(
    title="How the IRS Fails to Audit the Wealthiest Americans",
    author="Jesse Eisinger",
    word_count=5800,
    beat="federal tax policy"
)

# Use inherited methods
print(story.submit_for_review())
print(f"Corrections logged: {story.apply_correction(2)}")

'How the IRS Fails to Audit the Wealthiest Americans' submitted for editorial review — status: reviewed
Corrections logged: 2


In [5]:
# Use investigation-specific methods
print(story.file_foia("Internal Revenue Service"))
print(story.file_foia("Department of the Treasury"))
print(story.add_source("IRS audit rate data (FOIA response)"))
print(story.add_source("Whistleblower — former IRS senior examiner"))
print(story.add_source("Tax filings of 25 billionaires (leaked)"))
print(story.send_right_of_reply())
print(story.set_embargo(date(2024, 6, 8), "New York Times"))

📄 FOIA request filed with: Internal Revenue Service (1 total)
📄 FOIA request filed with: Department of the Treasury (2 total)
Source #1 recorded: IRS audit rate data (FOIA response)
Source #2 recorded: Whistleblower — former IRS senior examiner
Source #3 recorded: Tax filings of 25 billionaires (leaked)
✉️  Right of reply sent for: 'How the IRS Fails to Audit the Wealthiest Americans'
🔒 Embargo set until 2024-06-08 — co-publishing with New York Times


In [6]:
# Test publication rules — all conditions must be met
print("\n--- Publication attempt ---")
print(story.publish())    # blocked by embargo

# Lift the embargo by backdating for demo purposes
story.embargo_until = date(2024, 1, 1)
print(story.publish())    # should succeed now
print(f"Published: {story.is_published()}")
print(story)


--- Publication attempt ---
📰 Investigation published [federal tax policy]: 'How the IRS Fails to Audit the Wealthiest Americans' by Jesse Eisinger (co-published with New York Times)
📰 Investigation published [federal tax policy]: 'How the IRS Fails to Audit the Wealthiest Americans' by Jesse Eisinger (co-published with New York Times)
Published: True
Investigation: 'How the IRS Fails to Audit the Wealthiest Americans' [federal tax policy] — 3 source(s), 2 FOIA(s) — published


In [7]:
# Show what happens when rules are not respected
incomplete = Investigation(
    title="Medicare Billing Fraud in Rural Hospitals",
    author="Hannah Dreier",
    word_count=3400,
    beat="healthcare"
)
incomplete.submit_for_review()
incomplete.add_source("Only one source so far")
# right of reply not sent, only 1 source

print("\n--- Publishing without meeting requirements ---")
print(incomplete.publish())   # blocked: not enough sources

print(f"\nFormat metadata: {story.get_format()}")


--- Publishing without meeting requirements ---
❌ Cannot publish 'Medicare Billing Fraud in Rural Hospitals' — at least 2 verified sources required

Format metadata: {'type': 'Investigation', 'beat': 'federal tax policy', 'sources': 3, 'foia_requests': 2, 'right_of_reply': True, 'embargo_until': '2024-01-01', 'partner': 'New York Times', 'page_views': 0}


**Inheritance creates a type hierarchy.**

An `Investigation` is a `Content`. The `Investigation` class inherits `submit_for_review()`, `apply_correction()`, and status checking methods without duplication. It adds `file_foia()`, `send_right_of_reply()`, and `set_embargo()` for investigative workflows. It implements abstract methods `publish()` and `get_format()` with rules that are specific to ProPublica's journalism standards.

<Note type="important">

The `super()` function accesses parent methods. Calling `super().__init__(title, author, word_count)` invokes the parent constructor before adding investigation-specific initialization.

</Note>

## Polymorphism

Polymorphism enables writing code once that works with multiple types.

<img src = "https://data-analytics-fullstack-assets.s3.eu-west-3.amazonaws.com/M05-Python_Programming/M2_D3_Polymorphisme.png"/>

> same code, different behaviors depending on the object

You write code that works with **any object** that has the right methods, without caring about its exact class.

Example:

In [8]:
def go_live(content):
    """Duck-typed polymorphism: anything with .publish() works"""
    content.publish()

In [9]:
class AnInvestigation:
    def publish(self):
        print("Investigation published after source verification, FOIA review, and right of reply.")

class ADataReport:
    def publish(self):
        print("Data report published after methodology peer review and dataset validation.")


publish_queue = [AnInvestigation(), ADataReport()]

for content in publish_queue:
    go_live(content)

Investigation published after source verification, FOIA review, and right of reply.
Data report published after methodology peer review and dataset validation.


Same function `go_live(content)`:

* If `content` is an `AnInvestigation` → goes through full source and right-of-reply validation.
* If `content` is a `ADataReport` → goes through methodology peer review instead.

You didn't write `if type is AnInvestigation then ...`, `if type is ADataReport then ...`
You just said: **"I need something that has a `publish()` method."**
Each object does its own version of `publish()`.

In Python, that's often called **duck typing**:

> If it walks like a duck and quacks like a duck, I treat it like a duck. 🦆

In [10]:
class DataReport(Content):
    """Data-driven piece — lighter editorial process, stricter methodology rules

    DataReport inherits from Content.
    So it's a kind of Content, but built from datasets and analysis, not field reporting.
    It calls super().__init__ to reuse the initialization from Content.
    Then it adds its own specific stuff:
    - datasets: list of data sources used in the analysis
    - methodology_published: whether the methodology note is attached
    - peer_reviews: number of internal peer reviews completed
    So: same base workflow (review, status), plus data-specific validation attributes.
    """

    def __init__(self, title, author, word_count, analysis_tool="Python"):
        super().__init__(title, author, word_count)
        self.analysis_tool = analysis_tool     # e.g. 'Python', 'R', 'SQL'
        self.datasets = []                     # list of datasets used
        self.methodology_published = False     # methodology note must be attached
        self.peer_reviews = 0                  # internal peer reviews completed
        self.page_views = 0

    def add_dataset(self, name, source):
        """Register a dataset used in the analysis"""
        self.datasets.append({'name': name, 'source': source})
        return f"Dataset #{len(self.datasets)} registered: '{name}' from {source}"

    def attach_methodology(self):
        """Mark that the public methodology note has been written and attached"""
        self.methodology_published = True
        return f"📊 Methodology note attached to: '{self.title}'"

    def add_peer_review(self):
        """Record a completed internal peer review of the analysis"""
        self.peer_reviews += 1
        return f"✅ Peer review #{self.peer_reviews} completed for: '{self.title}'"

    def publish(self):
        """Implement abstract method — data reports require methodology and peer review"""
        if not self.is_ready():
            return f"❌ Cannot publish '{self.title}' — editorial review pending"
        if len(self.datasets) == 0:
            return f"❌ Cannot publish '{self.title}' — no datasets registered"
        if not self.methodology_published:
            return f"❌ Cannot publish '{self.title}' — methodology note required"
        if self.peer_reviews < 1:
            return f"❌ Cannot publish '{self.title}' — at least 1 peer review required"
        self.status = 'published'
        return (f"📊 Data report published: '{self.title}' by {self.author} "
                f"[{self.analysis_tool}, {len(self.datasets)} dataset(s)]")

    def get_format(self):
        """Implement abstract method — data report metadata"""
        return {
            'type': 'Data Report',
            'tool': self.analysis_tool,
            'datasets': len(self.datasets),
            'methodology': self.methodology_published,
            'peer_reviews': self.peer_reviews,
            'page_views': self.page_views
        }

    def __str__(self):
        return (f"Data Report: '{self.title}' [{self.analysis_tool}] "
                f"— {len(self.datasets)} dataset(s), {self.peer_reviews} review(s) "
                f"— {self.status}")

In [11]:
def daily_publish_run(contents: List[Content]):
    """
    Run ProPublica's daily publishing sequence across all content types.

    Accepts any Content objects. Each responds polymorphically.
    """
    print("=== PROPUBLICA — DAILY PUBLISH RUN ===")
    for content in contents:
        # Polymorphic call — publication rules vary entirely by content type
        print(content.publish())
    print("=" * 50)

Here:

* `daily_publish_run` doesn't know *which* kind of content it receives.
* It simply calls `content.publish()`.
* Each object runs **its own implementation** of `publish()`:

  * `Investigation` checks sources, FOIA records, right of reply, and embargo before going live,
  * `DataReport` checks datasets, methodology note, and peer review completion instead.

> Same function call, radically different editorial rules depending on the actual object type.
> That's polymorphism.

In [12]:
def newsroom_dashboard(contents: List[Content]):
    """Display the full editorial status and metadata for all content in the pipeline"""
    print("\n=== PROPUBLICA NEWSROOM DASHBOARD ===")
    for content in contents:
        print(f"{content}")
        fmt = content.get_format()
        print(f"  Metadata: {fmt}")
    print("=" * 50)

This function shows **two more polymorphic calls**:

* `print(f"{content}")` → calls `content.__str__()`, and each content class defines its own string representation — an `Investigation` shows its source and FOIA count, a `DataReport` shows its datasets and peer review status.
* `content.get_format()` → each content type returns entirely different metadata (sources, embargo, datasets, methodology, etc.) using the same method name.

> `newsroom_dashboard` only assumes that content items implement `__str__` and `get_format`.
> Add new content types tomorrow — newsletters, podcasts, live trackers — and it works unchanged.

That's the power of polymorphism:
**you write generic orchestration code once, and new classes plug in by respecting the same editorial interface.**

In [13]:
# Build today's content pipeline — a mix of Investigations and Data Reports
pipeline = [
    Investigation(
        title="How the IRS Fails to Audit the Wealthiest Americans",
        author="Jesse Eisinger",
        word_count=5800,
        beat="federal tax policy"
    ),
    Investigation(
        title="Solitary Confinement in Federal Prisons Since the Reform Act",
        author="Joaquin Sapien",
        word_count=4200,
        beat="criminal justice"
    ),
    DataReport(
        title="Mapping Hospital Billing Disparities Across U.S. States",
        author="Lena Groeger",
        word_count=1800,
        analysis_tool="Python"
    )
]

# Prepare Investigation #1 — fully meets all requirements
pipeline[0].submit_for_review()
pipeline[0].add_source("IRS audit rate data (FOIA response)")
pipeline[0].add_source("Whistleblower — former IRS senior examiner")
pipeline[0].add_source("Tax filings of 25 billionaires (leaked documents)")
pipeline[0].file_foia("Internal Revenue Service")
pipeline[0].send_right_of_reply()

# Prepare Investigation #2 — missing right of reply
pipeline[1].submit_for_review()
pipeline[1].add_source("Bureau of Justice Statistics report 2023")
pipeline[1].add_source("Interviews with 12 formerly incarcerated individuals")
pipeline[1].file_foia("Bureau of Prisons")
# right of reply not sent yet

# Prepare Data Report — meets all data requirements
pipeline[2].submit_for_review()
pipeline[2].add_dataset("CMS Hospital Price Transparency", "Centers for Medicare & Medicaid")
pipeline[2].add_dataset("State Medicaid Billing Records", "HHS FOIA request")
pipeline[2].attach_methodology()
pipeline[2].add_peer_review()

"✅ Peer review #1 completed for: 'Mapping Hospital Billing Disparities Across U.S. States'"

In [14]:
# Polymorphic operations work across all content types
daily_publish_run(pipeline)
newsroom_dashboard(pipeline)

=== PROPUBLICA — DAILY PUBLISH RUN ===
📰 Investigation published [federal tax policy]: 'How the IRS Fails to Audit the Wealthiest Americans' by Jesse Eisinger
❌ Cannot publish 'Solitary Confinement in Federal Prisons Since the Reform Act' — right of reply not yet sent
📊 Data report published: 'Mapping Hospital Billing Disparities Across U.S. States' by Lena Groeger [Python, 2 dataset(s)]

=== PROPUBLICA NEWSROOM DASHBOARD ===
Investigation: 'How the IRS Fails to Audit the Wealthiest Americans' [federal tax policy] — 3 source(s), 1 FOIA(s) — published
  Metadata: {'type': 'Investigation', 'beat': 'federal tax policy', 'sources': 3, 'foia_requests': 1, 'right_of_reply': True, 'embargo_until': None, 'partner': None, 'page_views': 0}
Investigation: 'Solitary Confinement in Federal Prisons Since the Reform Act' [criminal justice] — 2 source(s), 1 FOIA(s) — reviewed
  Metadata: {'type': 'Investigation', 'beat': 'criminal justice', 'sources': 2, 'foia_requests': 1, 'right_of_reply': False, 'e

In [15]:
# Each content type has specialized capabilities
print("\n=== SPECIALIZED OPERATIONS ===")
print(pipeline[0].set_embargo(date(2024, 9, 15), "Washington Post"))  # Investigation method
print(pipeline[2].add_peer_review())                                   # DataReport method


=== SPECIALIZED OPERATIONS ===
🔒 Embargo set until 2024-09-15 — co-publishing with Washington Post
✅ Peer review #2 completed for: 'Mapping Hospital Billing Disparities Across U.S. States'


<Note type ="tip">

Polymorphism delivers extensibility. `daily_publish_run()` and `newsroom_dashboard()` work with any `Content` subclass. Add a `Newsletter`, a `Podcast`, or a `LiveTracker` class tomorrow — as long as they implement `publish()` and `get_format()`, the orchestration code needs zero changes. The editorial rules live inside each class, not scattered across your pipeline functions.

</Note>

## Composition

Composition is a design principle where a class is composed of one or more objects from other classes, allowing for a flexible and modular structure.

<img src = "https://data-analytics-fullstack-assets.s3.eu-west-3.amazonaws.com/M05-Python_Programming/M2_D3_Composition.png"/>

At ProPublica, related investigations are grouped into **Series** — long-running thematic collections. For example, all pieces covering the IRS and tax inequality live in one `Series`. Each series has a lead journalist and accumulates content pieces over months or years.

When we say:

> A `Series` **has** a lead journalist and **has** multiple content pieces

we're saying:

* A **Series** is *not* a kind of Journalist.
* A **Series** is *not* a kind of Content.
* Instead, a **Series** is an object that **contains** a journalist and content pieces.

That's composition:

> building bigger objects by assembling smaller ones inside them.

In code:

In [16]:
class Journalist:
    def __init__(self, name, beat):
        self.name = name
        self.beat = beat   # e.g. 'criminal justice', 'federal tax policy', 'healthcare'


class ContentItem:
    def __init__(self, title, content_type):
        self.title = title
        self.content_type = content_type   # 'investigation', 'data report', 'interactive'


class Series:
    def __init__(self, name, lead_journalist):
        self.name = name
        self.lead_journalist = lead_journalist   # Series HAS a lead journalist
        self.pieces = []                          # Series HAS content pieces

    def add_piece(self, piece):
        self.pieces.append(piece)

    def remove_piece(self, title):
        self.pieces = [p for p in self.pieces if p.title != title]

Here:

* `Series` **has** a `Journalist` as its lead
* `Series` **has** a list of `ContentItem` objects

That's composition.

Composition becomes powerful when you combine it with **Polymorphism**. Because `Series` just **holds** content pieces, it doesn't care *what exact classes* they come from, as long as they behave correctly.

You can do things like:

In [17]:
class InvestigationItem(ContentItem):
    pass

class DataReportItem(ContentItem):
    pass

In [18]:
# A real ProPublica series — 'The Secret IRS Files'
lead = Journalist("Jesse Eisinger", "federal tax policy")

secret_irs_files = Series(
    name="The Secret IRS Files",
    lead_journalist=lead
)

secret_irs_files.add_piece(InvestigationItem(
    "The Secret IRS Files: Trove of Never-Before-Seen Records Reveal How the Wealthiest Avoid Income Tax",
    "investigation"
))
secret_irs_files.add_piece(DataReportItem(
    "We Ran the Numbers on Billionaire Tax Rates Since 2014",
    "data report"
))
secret_irs_files.add_piece(InvestigationItem(
    "Lord of the Roths: How Tech Mogul Peter Thiel Turned a Retirement Account for the Middle Class Into a $5 Billion Tax-Free Piggy Bank",
    "investigation"
))
secret_irs_files.add_piece(ContentItem(
    "Frequently Asked Questions About the Tax Data",
    "explainer"
))

In [19]:
print(f"📁 Series: {secret_irs_files.name}")
print(f"   Lead journalist: {secret_irs_files.lead_journalist.name}"
      f" (beat: {secret_irs_files.lead_journalist.beat})")
print(f"\n   Pieces ({len(secret_irs_files.pieces)}):")
for p in secret_irs_files.pieces:
    print(f"   [{p.content_type.upper()}] {p.title[:70]}...")

📁 Series: The Secret IRS Files
   Lead journalist: Jesse Eisinger (beat: federal tax policy)

   Pieces (4):
   [INVESTIGATION] The Secret IRS Files: Trove of Never-Before-Seen Records Reveal How th...
   [DATA REPORT] We Ran the Numbers on Billionaire Tax Rates Since 2014...
   [INVESTIGATION] Lord of the Roths: How Tech Mogul Peter Thiel Turned a Retirement Acco...
   [EXPLAINER] Frequently Asked Questions About the Tax Data...


`Series` works with:

* `InvestigationItem`
* `DataReportItem`
* `ContentItem`
* future formats — interactives, timelines, document viewers, audio stories

…because it's composed of generic `ContentItem` objects, not tied to a specific subclass. The series structure stays stable even as ProPublica experiments with new formats.

<Note type="tip">

* **Composition** = "has-a"

  * A Series **has** a lead journalist
  * A Series **has** content pieces

* **Inheritance** = "is-a"

  * An `Investigation` **is a** `Content`
  * A `DataReport` **is a** `Content`
  * An `InvestigationItem` **is a** `ContentItem`

Composition is great when:
* You're assembling systems from interchangeable parts
* You want to be able to **swap, add, or remove** pieces at runtime
* You want each part to evolve independently

</Note>

## Resources 📚📚

- [ProPublica website](https://www.propublica.org/)
- [Python Inheritance and Composition](https://realpython.com/inheritance-composition-python/)
- [Abstract Base Classes](https://docs.python.org/3/library/abc.html)
- [Special Method Names](https://docs.python.org/3/reference/datamodel.html#special-method-names)
- [Composition vs Inheritance](https://realpython.com/inheritance-composition-python/#whats-composition)
- [SOLID Principles](https://en.wikipedia.org/wiki/SOLID)